In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Reformat pivot_prcp_for_word_conus.csv for Supplementary Table layout:
- Drop the 3 metadata rows (window_type, trend_dir, domainName)
- Move CONUS to the first data row
- Reorder columns to match the table layout in your image:
  Ecoregion | 30 years (increase: PA + slope mn/mx/avg/md) | 30 years (decrease ...) |
            | 100 years (increase ...) | 100 years (decrease ...)
- Add a "Stat" column numbering rows starting from 1
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd


def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "scripts" else cwd


def find_project_root(marker: str) -> Path:
    base = get_project_root()
    required_path = base / marker
    if not required_path.exists():
        raise FileNotFoundError(f"Required project file not found: {required_path}")
    return base


def find_github_ready_dir() -> Path:
    return get_project_root()


def load_and_format_prcp_table(
    csv_path: str | Path,
    out_path: str | Path | None = None,
    conus_label: str = "CONUS",
) -> pd.DataFrame:
    csv_path = Path(csv_path)
    df_raw = pd.read_csv(csv_path)

    # ---------------------------------------------------------------------
    # 1) Drop the metadata rows produced by the pivot-for-Word export
    # ---------------------------------------------------------------------
    meta_markers = {"window_type", "trend_dir", "domainName"}
    df = df_raw[~df_raw["Unnamed: 0"].astype(str).isin(meta_markers)].copy()

    # Rename the domain column to match your table header
    df = df.rename(columns={"Unnamed: 0": "Ecoregion"})

    # ---------------------------------------------------------------------
    # 2) Move CONUS to the first row (if present)
    # ---------------------------------------------------------------------
    if (df["Ecoregion"] == conus_label).any():
        df_conus = df[df["Ecoregion"] == conus_label]
        df_rest = df[df["Ecoregion"] != conus_label]
        # keep the remaining ecoregions in their current order
        df = pd.concat([df_conus, df_rest], ignore_index=True)

    # ---------------------------------------------------------------------
    # 3) Reorder columns to match the image layout
    #    Your file columns map like this:
    #      recent_30yr increasing: PA, slope_min, slope_max, slope_mean, slope_median
    #      recent_30yr decreasing: PA.1, slope_min.1, slope_max.1, slope_mean.1, slope_median.1
    #      full_100yr increasing:  PA.2, slope_min.2, slope_max.2, slope_mean.2, slope_median.2
    #      full_100yr decreasing:  PA.3, slope_min.3, slope_max.3, slope_mean.3, slope_median.3
    # ---------------------------------------------------------------------
    rename_map = {
        # 30 years - increase
        "PA": "30y_inc_PA",
        "slope_min": "30y_inc_slope_mn",
        "slope_max": "30y_inc_slope_mx",
        "slope_mean": "30y_inc_slope_avg",
        "slope_median": "30y_inc_slope_md",
        # 30 years - decrease
        "PA.1": "30y_dec_PA",
        "slope_min.1": "30y_dec_slope_mn",
        "slope_max.1": "30y_dec_slope_mx",
        "slope_mean.1": "30y_dec_slope_avg",
        "slope_median.1": "30y_dec_slope_md",
        # 100 years - increase
        "PA.2": "100y_inc_PA",
        "slope_min.2": "100y_inc_slope_mn",
        "slope_max.2": "100y_inc_slope_mx",
        "slope_mean.2": "100y_inc_slope_avg",
        "slope_median.2": "100y_inc_slope_md",
        # 100 years - decrease
        "PA.3": "100y_dec_PA",
        "slope_min.3": "100y_dec_slope_mn",
        "slope_max.3": "100y_dec_slope_mx",
        "slope_mean.3": "100y_dec_slope_avg",
        "slope_median.3": "100y_dec_slope_md",
    }

    # Keep only columns that actually exist (safe if your CSV changes slightly)
    existing_rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=existing_rename_map)

    ordered_cols = [
        "Ecoregion",
        # 30 years
        "30y_inc_PA", "30y_inc_slope_mn", "30y_inc_slope_mx", "30y_inc_slope_avg", "30y_inc_slope_md",
        "30y_dec_PA", "30y_dec_slope_mn", "30y_dec_slope_mx", "30y_dec_slope_avg", "30y_dec_slope_md",
        # 100 years
        "100y_inc_PA", "100y_inc_slope_mn", "100y_inc_slope_mx", "100y_inc_slope_avg", "100y_inc_slope_md",
        "100y_dec_PA", "100y_dec_slope_mn", "100y_dec_slope_mx", "100y_dec_slope_avg", "100y_dec_slope_md",
    ]
    ordered_cols = [c for c in ordered_cols if c in df.columns]  # safety

    df = df[ordered_cols].copy()

    # ---------------------------------------------------------------------
    # 4) Add "Stat" numbering like your table (1,2,3,...)
    # ---------------------------------------------------------------------
    df.insert(0, "Stat", range(1, len(df) + 1))

    # ---------------------------------------------------------------------
    # 5) Optional save
    # ---------------------------------------------------------------------
    if out_path is not None:
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        if out_path.suffix.lower() in {".xlsx", ".xls"}:
            df.to_excel(out_path, index=False)
        else:
            df.to_csv(out_path, index=False)

    return df


if __name__ == "__main__":
    from pathlib import Path

    # ------------------------------------------------------------
    # INPUT / OUTPUT DIRS
    # ------------------------------------------------------------
    # For GitHub/reviewer use after data are copied into this folder:
    base_dir = find_project_root("data_intermediate/pivot_ai_for_word_conus.csv")
    in_dir = base_dir / "data_intermediate"   # where pivot_*_for_word_conus.csv lives
    out_dir = find_github_ready_dir() / "ST"                 # where you want the reordered files
    out_dir.mkdir(parents=True, exist_ok=True)

    # All files in your screenshot
    input_files = [
        "pivot_ai_for_word_conus.csv",
        "pivot_csdi_for_word_conus.csv",
        "pivot_pci_for_word_conus.csv",
        "pivot_prcp_for_word_conus.csv",
        "pivot_sdii_for_word_conus.csv",
        "pivot_tmax_for_word_conus.csv",
        "pivot_tmin_for_word_conus.csv",
        "pivot_wsdi_for_word_conus.csv",
    ]

    # ------------------------------------------------------------
    # PROCESS ALL
    # ------------------------------------------------------------
    for fname in input_files:
        in_csv = in_dir / fname

        # metric name for output file naming
        metric = fname.replace("pivot_", "").replace("_for_word_conus.csv", "")

        out_csv = out_dir / f"{metric}_supp_table_reordered.csv"
        # out_xlsx = out_dir / f"{metric}_supp_table_reordered.xlsx"  # optional

        df_formatted = load_and_format_prcp_table(in_csv, out_path=out_csv)
        # load_and_format_prcp_table(in_csv, out_path=out_xlsx)       # optional

        print(f"\nSaved: {out_csv}")
        print("Preview:")
        print(df_formatted.head(3).to_string(index=False))



In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Reformat pivot_prcp_for_word_conus.csv for Supplementary Table layout:
- Drop the 3 metadata rows (window_type, trend_dir, domainName)
- Move CONUS to the first data row
- Reorder columns to match the table layout in your image:
  Ecoregion | 30 years (increase: PA + slope mn/mx/avg/md) | 30 years (decrease ...) |
            | 100 years (increase ...) | 100 years (decrease ...)
- Add a "Stat" column numbering rows starting from 1
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd


def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "scripts" else cwd


def find_project_root(marker: str) -> Path:
    base = get_project_root()
    required_path = base / marker
    if not required_path.exists():
        raise FileNotFoundError(f"Required project file not found: {required_path}")
    return base


def find_github_ready_dir() -> Path:
    return get_project_root()


def load_and_format_prcp_table(
    csv_path: str | Path,
    out_path: str | Path | None = None,
    conus_label: str = "CONUS",
) -> pd.DataFrame:
    csv_path = Path(csv_path)
    df_raw = pd.read_csv(csv_path)

    # ---------------------------------------------------------------------
    # 1) Drop the metadata rows produced by the pivot-for-Word export
    # ---------------------------------------------------------------------
    meta_markers = {"window_type", "trend_dir", "domainName"}
    df = df_raw[~df_raw["Unnamed: 0"].astype(str).isin(meta_markers)].copy()

    # Rename the domain column to match your table header
    df = df.rename(columns={"Unnamed: 0": "Ecoregion"})

    # ---------------------------------------------------------------------
    # 2) Move CONUS to the first row (if present)
    # ---------------------------------------------------------------------
    if (df["Ecoregion"] == conus_label).any():
        df_conus = df[df["Ecoregion"] == conus_label]
        df_rest = df[df["Ecoregion"] != conus_label]
        # keep the remaining ecoregions in their current order
        df = pd.concat([df_conus, df_rest], ignore_index=True)

    # ---------------------------------------------------------------------
    # 3) Reorder columns to match the image layout
    #    Your file columns map like this:
    #      recent_30yr increasing: PA, slope_min, slope_max, slope_mean, slope_median
    #      recent_30yr decreasing: PA.1, slope_min.1, slope_max.1, slope_mean.1, slope_median.1
    #      full_100yr increasing:  PA.2, slope_min.2, slope_max.2, slope_mean.2, slope_median.2
    #      full_100yr decreasing:  PA.3, slope_min.3, slope_max.3, slope_mean.3, slope_median.3
    # ---------------------------------------------------------------------
    rename_map = {
        # 30 years - increase
        "PA": "30y_inc_PA",
        "slope_min": "30y_inc_slope_mn",
        "slope_max": "30y_inc_slope_mx",
        "slope_mean": "30y_inc_slope_avg",
        "slope_median": "30y_inc_slope_md",
        # 30 years - decrease
        "PA.1": "30y_dec_PA",
        "slope_min.1": "30y_dec_slope_mn",
        "slope_max.1": "30y_dec_slope_mx",
        "slope_mean.1": "30y_dec_slope_avg",
        "slope_median.1": "30y_dec_slope_md",
        # 100 years - increase
        "PA.2": "100y_inc_PA",
        "slope_min.2": "100y_inc_slope_mn",
        "slope_max.2": "100y_inc_slope_mx",
        "slope_mean.2": "100y_inc_slope_avg",
        "slope_median.2": "100y_inc_slope_md",
        # 100 years - decrease
        "PA.3": "100y_dec_PA",
        "slope_min.3": "100y_dec_slope_mn",
        "slope_max.3": "100y_dec_slope_mx",
        "slope_mean.3": "100y_dec_slope_avg",
        "slope_median.3": "100y_dec_slope_md",
    }

    # Keep only columns that actually exist (safe if your CSV changes slightly)
    existing_rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=existing_rename_map)

    ordered_cols = [
        "Ecoregion",
        # 30 years
        "30y_inc_PA", "30y_inc_slope_mn", "30y_inc_slope_mx", "30y_inc_slope_avg", "30y_inc_slope_md",
        "30y_dec_PA", "30y_dec_slope_mn", "30y_dec_slope_mx", "30y_dec_slope_avg", "30y_dec_slope_md",
        # 100 years
        "100y_inc_PA", "100y_inc_slope_mn", "100y_inc_slope_mx", "100y_inc_slope_avg", "100y_inc_slope_md",
        "100y_dec_PA", "100y_dec_slope_mn", "100y_dec_slope_mx", "100y_dec_slope_avg", "100y_dec_slope_md",
    ]
    ordered_cols = [c for c in ordered_cols if c in df.columns]  # safety

    df = df[ordered_cols].copy()

    # ---------------------------------------------------------------------
    # 4) Add "Stat" numbering like your table (1,2,3,...)
    # ---------------------------------------------------------------------
    df.insert(0, "Stat", range(1, len(df) + 1))

    # ---------------------------------------------------------------------
    # 5) Optional save
    # ---------------------------------------------------------------------
    if out_path is not None:
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        if out_path.suffix.lower() in {".xlsx", ".xls"}:
            df.to_excel(out_path, index=False)
        else:
            df.to_csv(out_path, index=False)

    return df


if __name__ == "__main__":
    from pathlib import Path

    # ------------------------------------------------------------
    # INPUT / OUTPUT DIRS
    # ------------------------------------------------------------
    # use after data are copied into this folder:
    base_dir = find_project_root("data_intermediate_spei_monthly/pivot_spei1_monthly_for_word_conus.csv")
    in_dir = base_dir / "data_intermediate_spei_monthly"   # where pivot_*_for_word_conus.csv lives
    out_dir = find_github_ready_dir() / "ST"                 # where you want the reordered files
    out_dir.mkdir(parents=True, exist_ok=True)

    # All files in your screenshot
    input_files = [
         "pivot_spei1_monthly_for_word_conus.csv",
        "pivot_spei3_monthly_for_word_conus.csv","pivot_spei6_monthly_for_word_conus.csv","pivot_spei12_monthly_for_word_conus.csv"
       
    ]

    # ------------------------------------------------------------
    # PROCESS ALL
    # ------------------------------------------------------------
    for fname in input_files:
        in_csv = in_dir / fname

        # metric name for output file naming
        metric = fname.replace("pivot_", "").replace("_for_word_conus.csv", "")

        out_csv = out_dir / f"{metric}_supp_table_reordered.csv"
        # out_xlsx = out_dir / f"{metric}_supp_table_reordered.xlsx"  # optional

        df_formatted = load_and_format_prcp_table(in_csv, out_path=out_csv)
        # load_and_format_prcp_table(in_csv, out_path=out_xlsx)       # optional

        print(f"\nSaved: {out_csv}")
        print("Preview:")
        print(df_formatted.head(3).to_string(index=False))

